# Detector C: fine-tune a prompt-injection classifier (Kaggle, GPU)

Run by the human (CLAUDE.md §4: the build machine has no CUDA GPU). Defensive use only: it trains on
the public `deepset/prompt-injections` dataset and nothing else.

**How to run**
1. Kaggle -> New Notebook -> upload this file -> Settings: Accelerator = GPU T4 x1, Internet = On.
2. Run all cells (about 5 minutes). It writes `/kaggle/working/outputs/predictions.jsonl` (scores for
   validation and test; **never** hard negatives, those are scored locally) and `outputs/model/`.
3. Download `outputs/` and run, locally:
   `uv run injection_bench import-predictions <downloaded folder>`.
4. To publish the model, see the last cell (a human action).

The splits are rebuilt exactly as `injection_bench data prepare` builds them (same revision, seed
and stratified 15 % validation carve) and **asserted against the expected id hashes**, so a mismatch
fails loudly instead of silently scoring different cases.

In [ ]:
import json, random, hashlib, os, sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# --- Pinned configuration (must match src/injection_bench/data/deepset.py) -------------------
DATASET_ID = "deepset/prompt-injections"
REVISION = "4f61ecb038e9c3fb77e21034b22511b523772cdd"
FILES = {
    "train": "data/train-00000-of-00001-9564e8b05b4757ab.parquet",
    "test": "data/test-00000-of-00001-701d16158af87368.parquet",
}
SEED = 42
VALIDATION_FRACTION = 0.15
EXPECTED_HASHES = {
    "train": "21e5a5ef10b11da8",
    "validation": "73bfb84584e09197",
    "test": "9c2c13a533988660",
}
EXPECTED_SIZES = {"train": 465, "validation": 81, "test": 116}

# --- Model / training choices (edit here; record any change in WORKLOG) -----------------------
MODEL_NAME = (
    "microsoft/deberta-v3-small"  # fallback for German: "distilbert-base-multilingual-cased"
)
MAX_LENGTH = 256
EPOCHS = 4
LEARNING_RATE = 3e-5
BATCH_SIZE = 16

OUT = Path("/kaggle/working/outputs") if Path("/kaggle/working").exists() else Path("outputs")
OUT.mkdir(parents=True, exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print("device:", "cuda" if torch.cuda.is_available() else "cpu", "| torch", torch.__version__)

In [ ]:
from huggingface_hub import hf_hub_download


def carve_validation(labels, fraction=VALIDATION_FRACTION, seed=SEED):
    rng = random.Random(seed)
    held = set()
    for label in (False, True):
        idx = [i for i, value in enumerate(labels) if value is label]
        rng.shuffle(idx)
        held.update(idx[: round(len(idx) * fraction)])
    return held


def ids_hash(ids):
    return hashlib.sha256("\n".join(sorted(ids)).encode()).hexdigest()[:16]


original_train = pd.read_parquet(
    hf_hub_download(DATASET_ID, FILES["train"], repo_type="dataset", revision=REVISION)
)
original_test = pd.read_parquet(
    hf_hub_download(DATASET_ID, FILES["test"], repo_type="dataset", revision=REVISION)
)
held = carve_validation([bool(v) for v in original_train["label"]])
frames = {
    "train": original_train.drop(index=list(held)).reset_index(drop=True),
    "validation": original_train.loc[sorted(held)].reset_index(drop=True),
    "test": original_test.reset_index(drop=True),
}
splits = {}
for name, frame in frames.items():
    ids = [f"ds-{name}-{i:04d}" for i in range(len(frame))]
    assert len(frame) == EXPECTED_SIZES[name], (name, len(frame))
    assert ids_hash(ids) == EXPECTED_HASHES[name], f"{name}: split ids differ from the local splits"
    splits[name] = {
        "ids": ids,
        "texts": frame["text"].astype(str).tolist(),
        "labels": frame["label"].astype(int).tolist(),
    }
print({k: len(v["ids"]) for k, v in splits.items()}, "splits match the local ones")

In [ ]:
from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "SAFE", 1: "INJECTION"},
    label2id={"SAFE": 0, "INJECTION": 1},
)


def encode(split):
    ds = Dataset.from_dict({"text": splits[split]["texts"], "label": splits[split]["labels"]})
    return ds.map(
        lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LENGTH),
        batched=True,
        remove_columns=["text"],
    )


train_ds, valid_ds = encode("train"), encode("validation")


def metrics(eval_pred):
    logits, labels = eval_pred
    pred = logits.argmax(-1)
    tp = int(((pred == 1) & (labels == 1)).sum())
    fp = int(((pred == 1) & (labels == 0)).sum())
    fn = int(((pred == 0) & (labels == 1)).sum())
    return {"recall": tp / max(tp + fn, 1), "precision": tp / max(tp + fp, 1)}


args = TrainingArguments(
    output_dir=str(OUT / "checkpoints"),
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=32,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=torch.cuda.is_available(),
    seed=SEED,
    report_to="none",
)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=metrics,
)
trainer.train()

In [ ]:
def score(split):
    ds = encode(split)
    logits = trainer.predict(ds).predictions
    z = logits - logits.max(axis=1, keepdims=True)
    p = np.exp(z) / np.exp(z).sum(axis=1, keepdims=True)
    return p[:, 1]


rows = []
for split in ("validation", "test"):  # never hard negatives: they stay on the laptop
    probs = score(split)
    labels = np.array(splits[split]["labels"])
    pred = probs >= 0.5
    print(
        f"{split}: n={len(labels)} recall@0.5={(pred & (labels == 1)).sum() / max((labels == 1).sum(), 1):.3f} "
        f"fpr@0.5={(pred & (labels == 0)).sum() / max((labels == 0).sum(), 1):.3f}"
    )
    rows += [
        {"case_id": cid, "split": split, "score": float(s)}
        for cid, s in zip(splits[split]["ids"], probs)
    ]

with open(OUT / "predictions.jsonl", "w", encoding="utf-8", newline="\n") as handle:
    for row in rows:
        handle.write(json.dumps(row) + "\n")

trainer.save_model(str(OUT / "model"))
tokenizer.save_pretrained(str(OUT / "model"))
(OUT / "run_info.json").write_text(
    json.dumps(
        {
            "model": MODEL_NAME,
            "epochs": EPOCHS,
            "lr": LEARNING_RATE,
            "batch_size": BATCH_SIZE,
            "max_length": MAX_LENGTH,
            "seed": SEED,
            "dataset_revision": REVISION,
            "torch": torch.__version__,
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        },
        indent=1,
    ),
    encoding="utf-8",
)
import shutil

shutil.rmtree(OUT / "checkpoints", ignore_errors=True)
print(sorted(p.name for p in OUT.iterdir()))

## Publish the model (human action)

Not done by the notebook. After `import-predictions`, from a logged-in machine:

```powershell
huggingface-cli login            # write token
huggingface-cli upload <hf-user>/injection-detector-deberta-small outputs/model .
```

Write the model card (training data, split, metrics with CIs, limitations, licence) first; record the
commit hash of the pushed revision in WORKLOG and pin it in `configs/` before the Space uses it.